# 04-llmlingua — 작은 모델이 토큰을 골라 버립니다

앞선 랩들과 판단 주체가 다릅니다.

| 랩 | 누가 판단하나 | 언어 영향 |
|---|---|---|
| `01` 무손실 | 규칙 | 없음 |
| `02` 참조핸들 | 겹침 점수 | 적음 |
| `03` 요약 | **큰 모델**이 다시 씁니다 | 적음 |
| **`04`** 프루닝 | **작은 모델**이 토큰마다 판정합니다 | **큽니다** |

중요도를 매기는 모델이 작아서, 그 모델이 약한 언어에서는 성능이 떨어집니다.
그래서 이 랩만 **한·영 이중언어 코퍼스**를 씁니다.

**이 노트북에서 확인할 것**

1. 세 변형이 같은 문장을 어떻게 다르게 줄이는가
2. 같은 설정에서 **한국어와 영어의 결과가 같은가**
3. 압축률을 높이면 어디서부터 답을 못 하게 되는가
4. 줄인 만큼 **비용과 시간이 실제로 줄어드는가**

## ⚠️ 처음 실행은 오래 걸립니다

모델을 내려받습니다. `v2` 약 700MB, `v1`/`long` 약 1GB 입니다.
이 랩은 **전용 가상환경**을 쓰므로 커널을 `labs/04-llmlingua/.venv` 로
잡아주세요.

## 1. kit 과 어댑터 불러오기

In [ ]:
import sys
from pathlib import Path

LAB = Path.cwd().resolve()
LABS = LAB.parents[0]                  # labs/<이 랩> -> labs
sys.path.insert(0, str(LABS))
sys.path.insert(0, str(LAB))           # 이 랩의 모듈(transforms, blocks 등)

# 노트북을 켜 둔 채로 저장소를 갱신하면 커널이 **예전 코드를 물고 있습니다.**
# 그러면 새로 생긴 함수가 없다는 에러(AttributeError)가 나는데, 원인이 코드가
# 아니라 커널이라 찾기가 어렵습니다. 그래서 이 셀을 돌릴 때마다 새로 읽습니다.
_stale = [m for m in list(sys.modules)
          if m == "kit" or m.startswith("kit.")
          or m in ("transforms", "blocks", "summarize", "compress")]
for _m in _stale:
    del sys.modules[_m]

from kit import VERSION, config as C, dataset, env, metrics, tokens as T
from kit.display import table, pct
from kit.runner import Run

# .env 는 labs/.env → 저장소 루트 .env → scripts/explore/.env 순으로 찾습니다.
env.load(verbose=True)

RUNS = LABS.parent / "runs"
print("kit", VERSION, "· 랩", LAB.name)
if _stale:
    print(f"모듈 {len(_stale)}개를 새로 읽었습니다 — 커널에 남아 있던 예전 코드를 지웠습니다")

import lingua as L
from compress import compress
from kit.metrics import survival

table(
    ["변형", "무엇이 다른가", "필요한 것"],
    [["v1", "토큰별 정보량으로 프루닝", "인과 LM"],
     ["long", "질문을 주고 문단별 중요도를 함께 봄", "인과 LM + 질문"],
     ["v2", "분류 모델이 토큰을 남길지 판정", "전용 인코더"]],
    align=["left", "left", "left"],
    title="LLMLingua 3형제 — 같은 클래스, 다른 파라미터",
)

# 모델은 크기별로 고를 수 있습니다. 기본은 small 입니다.
table(
    ["변형", "별칭", "모델", "크기"],
    [[v, tier, name.split("/")[-1][:42], size]
     for v, tiers in L.MODELS.items() for tier, (name, size) in tiers.items()],
    align=["left", "left", "left", "left"],
    title="고를 수 있는 모델",
    note="기본은 small 입니다 — 받자마자 돌려보실 수 있어야 해서입니다. "
         "config 의 model_name 이나 --model 로 바꾸실 수 있습니다.",
)

## 1-1. 어느 모델로 돌릴지 고르기

아래 셀을 실행하면 드롭다운이 나옵니다. **바꾸신 뒤 아래 셀들을 다시
실행하면** 그 모델로 결과가 나옵니다.

| 별칭 | `v1`·`long` | `v2` | 특징 |
|---|---|---|---|
| `small` (기본) | Qwen2.5-0.5B · 1GB | bert-base-multilingual · 700MB | 빠릅니다 |
| `large` | Qwen2.5-1.5B · 3GB | **xlm-roberta-large · 2.2GB** | 정확합니다 |
| `paper` | Llama-2-7b · 13GB | — | 논문이 쓴 것 |

**처음 고르시는 모델은 내려받느라 몇 분 걸립니다.** `~/.cache/huggingface/`
에 남아 다음부터는 로딩만 합니다.

> 드롭다운이 안 보이면 `ipywidgets` 가 없는 것입니다. 그때는 아래 셀의
> `MODEL = "small"` 을 직접 고치셔도 똑같이 동작합니다.

In [ ]:
# 이 값을 바꾸면 아래 셀들이 전부 그 모델을 씁니다.
MODEL = "small"          # small | large | paper | HuggingFace 경로

try:
    import ipywidgets as W
    from IPython.display import display

    _sel = W.Dropdown(
        options=[("small — 빠릅니다 (기본)", "small"),
                 ("large — 정확합니다", "large"),
                 ("paper — 논문이 쓴 모델 (v1/long 만, 13GB)", "paper")],
        value=MODEL, description="모델:",
        style={"description_width": "initial"},
        layout=W.Layout(width="420px"))

    def _on(change):
        global MODEL
        MODEL = change["new"]
        print(f"MODEL = {MODEL!r} · 아래 셀들을 다시 실행해 주세요")

    _sel.observe(_on, names="value")
    display(_sel)
except ImportError:
    print("ipywidgets 가 없어 드롭다운을 못 만듭니다.")
    print("위의 MODEL 값을 직접 고치시면 똑같이 동작합니다.")


def model_for(variant):
    """고른 모델을 이 변형에 맞게 풀어 줍니다.

    v2 에는 paper 티어가 없습니다. 그때는 large 로 대신합니다 —
    조용히 small 로 떨어지면 "큰 모델을 골랐는데 결과가 그대로" 가 됩니다.
    """
    tiers = L.MODELS[variant]
    if MODEL in tiers:
        return MODEL
    if MODEL == "paper" and "large" in tiers:
        print(f"  ({variant} 에는 paper 가 없어 large 로 대신합니다)")
        return "large"
    return MODEL          # HuggingFace 경로를 직접 주신 경우


print(f"\n현재 MODEL = {MODEL!r}")
for v in ("v1", "long", "v2"):
    print(f"  {v:5s} → {L.resolve_model(v, model_for(v))}")

## 지표 두 가지 — 표를 읽기 전에

| 이름 | 무엇을 재나 |
|---|---|
| **절감** | 토큰이 얼마나 줄었나 |
| **보존율** | 답에 꼭 필요한 문자열(`must_include`)이 압축 후에도 남은 비율 |

보존율 예시입니다.

```
질문        3월 결제 총액과 환불액은?
필요한 것    ["32,450,000", "1,280,500"]   ← 2개

압축 후 2개 다 남음  → 100%
1개만 남음          →  50%
```

아래 표에서 **`전체` · `한국어` · `영어` 는 전부 같은 보존율**입니다.
전체는 12건 평균이고, 나머지는 그중 해당 언어만 골라 낸 평균입니다.

> 지금은 한·영이 6건씩 같아서 전체가 두 언어의 가운데값과 일치합니다.
> **건수가 달라지면 많은 쪽으로 기웁니다.** 한국어 서비스에 쓰실 거라면
> 전체가 아니라 **한국어 열**을 기준으로 보세요.

## 2. 코퍼스 — 같은 사실을 한국어와 영어로

번역이 아니라 **같은 사실을 담은 쌍**입니다. 한국어를 기계번역하면 어색한
문장이 나와서, 각 언어로 자연스럽게 따로 썼습니다. `pair_id` 로 묶여 있어
`ko-01` 과 `en-01` 을 짝지어 볼 수 있습니다.

유형은 앞선 랩들과 같습니다. 압축이 잘 깨지는 자리를 골라 둔 것입니다.

| 유형 | 무엇을 겨냥하나 |
|---|---|
| `numeric` | 금액·비율이 답인 경우 |
| `negation` | 부정어가 사라지면 뜻이 뒤집힙니다 |
| `identifier` | 코드·사번처럼 문맥 없이 떠 있는 값 |
| `structured` | 표에 가까운 나열 |
| `longdoc` | 여러 문단. 프루닝이 이득을 내려면 길이가 필요합니다 |

아래 셀은 언어별 분량과 **문자당 토큰 수**를 봅니다. 같은 내용을 담는 데
각 언어가 토큰을 얼마나 쓰는지 먼저 확인해 두는 것입니다.

In [ ]:
cases = dataset.load("../data/sample-bilingual")
counter = T.make_counter({"mode": "local"}, "gpt-5.4")

ko = [c for c in cases if c.meta["lang"] == "ko"]
en = [c for c in cases if c.meta["lang"] == "en"]

table(
    ["언어", "건수", "문자", "토큰", "문자당 토큰"],
    [[lg, len(xs), f"{sum(len(c.text) for c in xs):,}",
      f"{sum(counter(c.text) for c in xs):,}",
      f"{sum(counter(c.text) for c in xs) / sum(len(c.text) for c in xs):.2f}"]
     for lg, xs in [("한국어", ko), ("영어", en)]],
    align=["left", "right", "right", "right", "right"],
    title="이중언어 코퍼스",
    note="문자당 토큰이 언어마다 다릅니다. 뒤의 절감률을 읽으실 때 "
         "이 값을 기억해 두세요 — 같은 절감률이라도 아끼는 토큰 수가 다릅니다.",
)

c = ko[0]
print(f"[{c.id}] {c.question}")
print(f"  {c.text[:70]}…")
print(f"  정답 문자열 {c.must_include}")

## 3. 세 변형이 같은 문장을 어떻게 다루나 — 한국어와 영어로

### 세 변형이 무엇인지 먼저

이름은 비슷하지만 **무엇을 근거로 버릴지**가 다릅니다.

| 변형 | 무엇을 보고 버릴 토큰을 정하나 | 질문을 쓰나 |
|---|---|---|
| **`v1`** LLMLingua | 인과 LM 이 매긴 **토큰별 정보량**. 예측하기 쉬운 토큰부터 버립니다 | ✗ |
| **`long`** LongLLMLingua | 위와 같되, **질문과의 관련도**로 문단 순위를 먼저 매깁니다 | **✓** |
| **`v2`** LLMLingua-2 | 전용 분류 모델이 토큰마다 **남길지 말지 직접 판정**합니다 | ✗ |

`v1` 과 `long` 은 "이 토큰이 얼마나 놀라운가" 를 보고, `v2` 는 "이 토큰을
남겨야 하나" 를 학습한 대로 답합니다. 앞의 둘은 인과 LM 이 필요하고
`v2` 는 전용 인코더를 씁니다.

### 무엇을 확인하나

**같은 사실을 담은 한·영 쌍**(`ko-01` / `en-01`)에 셋을 각각 겁니다.
언어만 다르고 내용·질문·정답 문자열이 같으므로, 결과가 갈리면 그건
순전히 언어 때문입니다.

표에서 이렇게 보세요.

- **절감** — 얼마나 줄였나
- **보존율** — 답에 필요한 문자열이 남았나
- **남은 정답 문자열** — 무엇이 남고 무엇이 사라졌나

절감이 크면서 보존율도 높은 조합이 있는지, 아니면 맞바꿔야 하는지 보시면
됩니다.

**처음 실행하면 모델 세 개를 받느라 몇 분 걸립니다.**

In [ ]:
# ── 이 셀의 설정 ──────────────────────────────────────────────
RATE = 0.5               # 남길 비율. 낮출수록 많이 버립니다
RESERVE_DIGIT = True     # 숫자를 지키려 시도합니다
PAIR = ["ko-01", "en-01"]

# 같은 사실을 담은 한·영 쌍입니다. 언어만 다르고 내용·질문·정답이 같습니다.
print(f"설정 · 모델 {MODEL} · rate {RATE} · force_reserve_digit {RESERVE_DIGIT}")
print()

for cid in PAIR:
    probe = [c for c in cases if c.id == cid][0]
    print(f"[{cid}] {probe.question}")
    print(f"  원문: {probe.text[:88]}…")
    print(f"  정답 문자열: {probe.must_include}")
    print()

rows = []
for cid in PAIR:
    probe = [c for c in cases if c.id == cid][0]
    for v in ["v1", "long", "v2"]:
        out, meta = compress(probe.text, question=probe.question, variant=v,
                             model_name=model_for(v),
                             rate=RATE, force_reserve_digit=RESERVE_DIGIT)
        kept = [m for m in probe.must_include
                if m.replace(",", "") in out.replace(" ", "").replace(",", "")]
        rows.append([cid, v, f"{counter(probe.text)} → {counter(out)}",
                     pct(1 - counter(out) / counter(probe.text)),
                     pct(survival(out, probe.must_include)),
                     ", ".join(kept) if kept else "(전부 사라짐)"])
        print(f"  {cid} / {v} 완료", flush=True)

table(
    ["케이스", "변형", "토큰", "절감", "보존율", "남은 정답 문자열"],
    rows,
    align=["left", "left", "right", "right", "right", "left"],
    title=f"세 변형 비교 · 모델 {MODEL} · rate={RATE} "
          f"· force_reserve_digit={RESERVE_DIGIT}",
    note="세 줄씩 두 묶음입니다. 같은 행의 한국어·영어를 짝지어 보시고, "
         "'남은 정답 문자열' 이 비어 있으면 그 질문에는 답할 수 없습니다.",
)

### 표를 읽으실 때

**절감률과 보존율을 반드시 같이 보세요.** 한쪽만 보면 뒤집힌 결론이 나옵니다.

| 이런 조합이면 | 뜻 |
|---|---|
| 절감 ≈ 0% · 보존 100% | 압축을 거의 안 한 것입니다. 잘한 게 아닙니다 |
| 절감 큼 · 보존 0% | 많이 줄였지만 답할 근거가 사라졌습니다 |
| 절감 큼 · 보존 100% | 이 조건에서는 쓸 만합니다 |

그리고 세 가지를 확인해 보세요.

1. **같은 행의 한국어·영어가 같은 결과인가** — 갈린다면 내용이 아니라
   언어 때문입니다. 내용·질문·정답이 같은 쌍이니까요.
2. **'남은 정답 문자열' 에 무엇이 사라졌나** — 금액인지 식별자인지에 따라
   위험도가 다릅니다.
3. **압축 결과를 직접 읽어보면** 숫자가 온전한지 중간에서 끊겼는지
   보입니다. 아래 셀로 원문과 나란히 볼 수 있습니다.

> 결과는 **고른 모델과 `rate` 에 따라 달라집니다.** 위의 드롭다운을 바꿔
> 다시 돌려보시면 같은 코드로 다른 그림이 나옵니다.

In [ ]:
# 압축 결과를 눈으로 확인합니다. 표의 숫자만으로는 "어떻게" 깨졌는지
# 알 수 없습니다.
LOOK = "en-01"          # 보고 싶은 케이스로 바꾸셔도 됩니다

probe = [c for c in cases if c.id == LOOK][0]
print(f"[{LOOK}] 모델 {MODEL} · rate {RATE}")
print(f"질문: {probe.question}")
print(f"정답 문자열: {probe.must_include}")
print()
print("원문")
print(" ", probe.text[:150])

for v in ["v1", "long", "v2"]:
    out, _ = compress(probe.text, question=probe.question, variant=v,
                      model_name=model_for(v), rate=RATE,
                      force_reserve_digit=RESERVE_DIGIT)
    print()
    print(v)
    print(" ", out[:150].replace(chr(10), " "))

## 4. 조용히 무시되는 인자 — 이 랩에서 가장 조심할 부분

`use_llmlingua2=True` 인 압축기에 `question` 이나 `rank_method` 를 넘기면
**에러 없이 무시됩니다.** LongLLMLingua 설정을 v2 에 잘못 붙여도 그냥
돌아가고 결과만 v2 그대로입니다.

숫자가 안 바뀌는 이유를 찾느라 시간을 버리기 쉬워서, 어댑터가 거부합니다.

In [ ]:
try:
    L.check_params("v2", {"rate": 0.5, "question": "환불 수수료는?",
                          "rank_method": "longllmlingua"})
    print("통과 — 이러면 안 됩니다")
except ValueError as e:
    print("거부됨")
    print()
    print(e)

## 5. 이 랩의 모든 조건 돌려보기

**조건 1개 = 파일 1개**입니다. `configs/` 를 훑으면 이 랩이 답할 수 있는
질문이 전부 나옵니다. 설정을 새로 추가해도 이 셀은 고칠 필요가 없습니다.

각 조건은 `runs/04-llmlingua/<설정이름>/<시각>/` 에 따로 기록됩니다. 나중에
"그때 무엇을 돌렸나" 를 설정 이름만 보고 알 수 있게 하려는 것입니다.

| 설정 | 무엇을 보려고 |
|---|---|
| `v2` | 이 랩의 권장 조건 |
| `v1` | 질문 없이 토큰 정보량만 |
| `long` | 질문을 주면 나아지나 |
| `v2-noop` | `rate=1.0` **자가 점검** |

`v2-noop` 이 중요합니다. 아무것도 안 버리는 설정인데 **토큰이 늘어납니다.**

In [ ]:
def run_config(path):
    cfg = C.load(path)
    cs = dataset.load(cfg.dataset["path"], limit=cfg.dataset.get("limit"))
    cnt = T.make_counter({"mode": "local"}, cfg.model)
    params = dict(cfg.params)
    variant = params.pop("variant", "v2")
    # 설정 파일이 지정한 모델을 그대로 씁니다. 위에서 고른 MODEL 은
    # 개별 셀에만 적용되고, 여기서는 config 가 기준입니다.
    mdl = params.pop("model_name", None)

    run = Run(cfg, RUNS)
    for x in cs:
        after, extra = compress(x.text, question=x.question or "",
                                variant=variant, model_name=mdl, **params)
        extra["lang"] = x.meta.get("lang", "-")
        run.add(metrics.per_case(x.id, x.kind, x.text, after,
                                 x.must_include, cnt, extra),
                before=x.text, after=after)

    m = metrics.aggregate(run.records, cnt)
    m["variant"] = variant
    m["config"] = cfg.name
    m["model"] = L.resolve_model(variant, mdl)
    m["rate"] = params.get("rate")
    for lg in ("ko", "en"):
        xs = [r for r in run.records if r["lang"] == lg]
        if xs:
            m[f"surv_{lg}"] = sum(r["survival"] for r in xs) / len(xs)
    return cfg, m, run.finish(m, [f"변형 {variant}"])


results = []
for p in sorted(Path("configs").glob("*.yaml")):
    cfg, m, out = run_config(p)
    results.append((cfg.name, m, out))
    print(f'{cfg.name:12s} {m["variant"]:5s} rate={str(m["rate"]):4s} '
          f'{m["model"].split("/")[-1][:34]:36s} '
          f'절감 {m["saved"]:6.1%} · 보존 {m["survival_mean"]:6.1%}', flush=True)

## 6. 조건 비교

같은 코드에 조건만 바꿔 돌린 결과입니다. **숫자 하나가 아니라 표를 보세요.**
어떤 조건에서 무엇을 얻고 무엇을 잃는지가 이 랩의 결론입니다.

In [ ]:
n_ko = len([c for c in cases if c.meta["lang"] == "ko"])
n_en = len([c for c in cases if c.meta["lang"] == "en"])

# 뒤의 세 열은 같은 보존율을 언어별로 나눈 것입니다. 열 이름에 건수를 적어
# "전체가 두 언어의 단순평균인가?" 라는 오해를 줄입니다.
rows = []
for n, m, _ in results:
    ko, en = m.get("surv_ko"), m.get("surv_en")
    gap = (en - ko) if (ko is not None and en is not None) else None
    rows.append([n, m["variant"], m["model"].split("/")[-1][:26], m["rate"],
                 pct(m["saved"]),
                 pct(m["survival_mean"]), pct(ko), pct(en),
                 "—" if gap is None else f"{gap * 100:+.1f}%p"])

table(
    ["설정", "변형", "모델", "rate", "절감",
     f"보존율 전체({len(cases)}건)", f"한국어({n_ko}건)", f"영어({n_en}건)",
     "영−한 격차"],
    rows,
    align=["left", "left", "left", "right", "right",
           "right", "right", "right", "right"],
    title="조건 비교 — 설정마다 변형·모델·rate 가 다릅니다",
    note=f"'전체' 는 {len(cases)}건 전부의 평균입니다. 지금은 한·영이 {n_ko}건씩 "
         f"같아서 두 언어 평균의 가운데값과 일치하지만, 건수가 달라지면 "
         f"많은 쪽으로 기웁니다.",
)

real = [m for _, m, _ in results if m.get("rate") != 1.0]
if real:
    w = max(real, key=lambda m: (m.get("surv_en") or 0) - (m.get("surv_ko") or 0))
    print(f"격차가 가장 큰 조건은 {w['config']} 입니다.")
    print(f"  전체 {w['survival_mean']:.1%} 로는 무난해 보이지만 "
          f"한국어만 보면 {w['surv_ko']:.1%} 입니다.")
    print("한국어 서비스에 쓰신다면 '전체' 가 아니라 '한국어' 열을 보세요.")

by = {n: m for n, m, _ in results}
if "v2-noop" in by:
    m = by["v2-noop"]
    print(f'v2-noop: rate=1.0 인데 절감 {m["saved"]:.1%} · '
          f'보존율 {m["survival_mean"]:.1%}')
    print("아무것도 안 버렸는데 토큰이 늘었습니다. 토큰에서 텍스트를 다시")
    print("만들면서 '32,450,000' 이 '32, 450, 000' 처럼 벌어지기 때문입니다.")
if "v2" in by and "long" in by:
    print()
    print(f'v2 보존 {by["v2"]["survival_mean"]:.1%} vs '
          f'long 보존 {by["long"]["survival_mean"]:.1%} — '
          f'질문을 주는 long 이 오히려 나쁩니다.')
    print("작은 모델로 토큰 단위 프루닝을 하면 숫자가 조각나기 때문입니다.")

## 7. 압축률을 바꿔가며 — 절감과 보존율의 관계

지금까지는 `rate=0.5` 한 점만 봤습니다. 여기서는 `rate` 를 0.9 부터 0.3 까지
낮춰가며 **절감률과 보존율이 어떻게 함께 움직이는지** 봅니다.

`rate` 는 **남길 비율**입니다. 0.9 면 10% 만 버리고, 0.3 이면 70% 를 버립니다.

표에서 이렇게 보세요.

- 절감률이 오를 때 보존율이 **어느 지점부터** 떨어지기 시작하는가
- 한국어 열과 영어 열이 **같이 움직이는가, 갈라지는가**
- 쓸 만한 보존율을 유지하면서 얻을 수 있는 절감률의 상한은 얼마인가

In [ ]:
# ── 이 셀의 설정 ──────────────────────────────────────────────
RATES = [0.9, 0.7, 0.5, 0.3]     # 남길 비율
RESERVE_DIGIT = True

print(f"설정 · 변형 v2 · 모델 {MODEL} · rate {RATES} "
      f"· force_reserve_digit {RESERVE_DIGIT}")
print()

rows = []
for rate in RATES:
    recs = []
    for x in cases:
        out, meta = compress(x.text, variant="v2", model_name=model_for("v2"),
                             rate=rate, force_reserve_digit=RESERVE_DIGIT)
        recs.append({"lang": x.meta["lang"],
                     "s": survival(out, x.must_include),
                     "tb": counter(x.text), "ta": counter(out)})
    tb = sum(r["tb"] for r in recs)
    ta = sum(r["ta"] for r in recs)
    g = {lg: [r["s"] for r in recs if r["lang"] == lg] for lg in ("ko", "en")}
    rows.append([rate, pct(1 - ta / tb),
                 pct(sum(r["s"] for r in recs) / len(recs)),
                 pct(min(r["s"] for r in recs)),
                 pct(sum(g["ko"]) / len(g["ko"])),
                 pct(sum(g["en"]) / len(g["en"]))])
    print(f"  rate={rate} 완료", flush=True)

table(
    ["rate", "절감", f"보존 전체({len(cases)}건)", "보존 최저",
     f"한국어({n_ko}건)", f"영어({n_en}건)"],
    rows,
    align=["right"] * 6,
    title=f"압축률 스윕 · 변형 v2 · 모델 {MODEL} "
          f"· force_reserve_digit={RESERVE_DIGIT}",
    note="위에서 아래로 갈수록 많이 버립니다. 한국어와 영어 열이 갈라지기 "
         "시작하는 지점이 어디인지, 그때 절감률이 얼마인지 보세요.",
)

## 8. 레이턴시 트레이드오프 — 압축은 공짜가 아닙니다

여기까지는 **얼마나 줄었나**만 봤습니다. 그런데 압축 자체도 시간을 씁니다.
작은 모델을 한 번 더 돌리는 일이니까요.

그래서 세 조건을 **끝에서 끝까지** 재봅니다.

| 재는 것 | 무엇 |
|---|---|
| **압축 소요** | LLMLingua 가 텍스트를 줄이는 데 걸린 시간 |
| **응답 소요** | 그 컨텍스트로 본 모델에 물어보고 답을 받기까지 |
| **합계** | 사용자가 실제로 기다리는 시간 |

모델 로딩은 미리 끝내고 잽니다. 한 번만 드는 비용이라 매 요청에 포함하면
오해를 부릅니다.

> 긴 케이스(`longdoc`·`structured`)만 씁니다. 짧은 글은 압축할 것도 없고
> 시간 차이도 묻힙니다.

In [ ]:
import time
import statistics as st
from kit.provider import complete

# ── 이 셀의 설정 ──────────────────────────────────────────────
RATE = 0.5
RESERVE_DIGIT = True
KINDS = ("longdoc", "structured")   # 긴 것만. 짧은 글은 시간 차이가 묻힙니다
TIERS = ("none", "small", "large")

DEP = env.get("AZURE_OPENAI_DEPLOYMENT")
use = [c for c in cases if c.meta["kind"] in KINDS]
print(f"설정 · 변형 v2 · rate {RATE} · 유형 {KINDS} · 본 모델 {DEP}")

for tier in ("small", "large"):
    L.load("v2", tier)          # 로딩을 먼저 끝냅니다. 측정에 섞이면 안 됩니다.
print(f"케이스 {len(use)}건 · 모델 로딩 완료\n")


def ask(ctx, q):
    t0 = time.perf_counter()
    complete(f"{ctx}\n\n질문: {q}\n한 문장으로 짧게 답하세요.",
             DEP, max_output_tokens=256)
    return time.perf_counter() - t0


rows, detail = [], {}
for cond in TIERS:
    recs = []
    for c in use:
        if cond == "none":
            comp_ms, out = 0.0, c.text
        else:
            t0 = time.perf_counter()
            out, _ = compress(c.text, variant="v2", model_name=cond,
                              rate=RATE, force_reserve_digit=RESERVE_DIGIT)
            comp_ms = (time.perf_counter() - t0) * 1000
        recs.append(dict(lang=c.meta["lang"], comp_ms=comp_ms, lat=ask(out, c.question),
                         tb=counter(c.text), ta=counter(out),
                         s=survival(out, c.must_include)))
    detail[cond] = recs

    tb, ta = sum(r["tb"] for r in recs), sum(r["ta"] for r in recs)
    ko = [r["s"] for r in recs if r["lang"] == "ko"]
    en = [r["s"] for r in recs if r["lang"] == "en"]
    cm = st.median(r["comp_ms"] for r in recs)
    lt = st.median(r["lat"] for r in recs)
    rows.append([{"none": "압축 없음", "small": "v2 small", "large": "v2 large"}[cond],
                 f"{tb:,} → {ta:,}", pct(1 - ta / tb),
                 pct(sum(ko) / len(ko)), pct(sum(en) / len(en)),
                 f"{cm:.0f}ms", f"{lt:.2f}s", f"{cm / 1000 + lt:.2f}s"])
    print(f"  {cond} 완료", flush=True)

table(
    ["조건", "토큰", "절감", "보존 한국어", "보존 영어",
     "압축 소요", "응답 소요", "합계"],
    rows,
    align=["left", "right", "right", "right", "right", "right", "right", "right"],
    title=f"끝에서 끝까지 · 변형 v2 · rate={RATE} · 케이스 {len(use)}건 "
          f"· 본 모델 {DEP}",
    note="'합계' 가 사용자가 기다리는 시간입니다. 중앙값이며 응답 시간은 "
         "호출마다 흔들리므로 소수점 아래는 잡음으로 보세요.",
)

### 표를 읽으실 때

**`합계` 열을 `압축 없음` 행과 비교하세요.** 그게 사용자가 실제로 기다리는
시간입니다. 토큰이 줄었다고 합계가 자동으로 줄지는 않습니다.

세 가지를 확인해 보세요.

1. **압축 소요가 응답 소요에 비해 어느 정도인가** — 응답이 훨씬 크면
   압축 시간은 묻히고, 비슷하면 그대로 손해로 잡힙니다.
2. **입력이 절반으로 줄었을 때 응답 소요도 그만큼 줄었는가** — 안 줄었다면
   그 규모에서는 응답 시간이 입력 길이에 좌우되지 않는다는 뜻입니다.
3. **`small` 과 `large` 의 압축 소요 차이** — 큰 모델은 정확한 대신 느립니다.

> 응답 소요는 호출마다 흔들립니다. 한 번의 차이는 잡음일 수 있으니,
> 판단이 필요하시면 셀을 두세 번 돌려보세요.

### 언제 속도에도 이득이 나나

입력이 커서 **프리필이 병목**이 될 때입니다. 손익은 이렇게 갈립니다.

```
압축으로 아끼는 시간  ≈  (줄인 토큰 수) × (토큰당 프리필 시간)
압축에 쓰는 시간      ≈  위 표의 '압축 소요'

앞이 뒤보다 커야 속도로도 이득입니다.
```

컨텍스트가 짧으면 왼쪽이 작아서 이기기 어렵고, 수만 토큰이면 왼쪽이
커집니다. **본인 워크로드의 실제 입력 길이로 이 셀을 돌려보시는 것**이
가장 확실합니다.

## 9. 진짜로 줄었나 — API 응답으로 확인하기

여기까지의 절감률은 전부 **tiktoken 추정치**입니다. 실제로 청구되는 값은
API 응답의 `usage.input_tokens` 이고, 둘이 항상 같지는 않습니다.

| 왜 어긋나나 | 얼마나 |
|---|---|
| 메시지 포맷 오버헤드 (역할 구분자 등) | 텍스트당 상수 (실측 +6) |
| 배포 모델의 토크나이저가 tiktoken 과 다를 수 있음 | 모델마다 |
| 압축 결과의 특수 문자를 모델이 어떻게 쪼개는지 | **해봐야 압니다** |

마지막 줄이 중요합니다. 이 랩은 **압축 결과를 토큰에서 다시 만듭니다.** 그 과정에서 `32,450,000` 이 `32, 450, 000` 으로 벌어지기도 하는데, 그렇게 바뀐 글자를 모델 토크나이저가 어떻게 쪼개는지는 tiktoken 추정으로 알 수 없습니다. `rate=1.0` 인데 토큰이 늘었던 것도 같은 이유였습니다.

그래서 몇 건만 뽑아 **압축 전과 후를 각각 실제로 보내보고**, 응답이 알려주는
토큰 수로 절감률을 다시 계산합니다. 호출은 케이스당 2회이고 캐시됩니다.

In [ ]:
from kit import verify

DEPLOY = env.get("AZURE_OPENAI_DEPLOYMENT")
BILLED = None                     # 아래 리포트에서 다시 씁니다

try:
    cfg4 = C.load("configs/v2.yaml")
    cs4 = dataset.load(cfg4.dataset["path"])
    params4 = dict(cfg4.params)
    params4.pop("variant", None)
    params4.pop("model_name", None)

    # 코퍼스가 ko/en 교대라 앞에서부터 4건을 뽑으면 두 언어가 2건씩 들어옵니다.
    # 3건만 뽑으면 한국어로 기울어 언어 비교가 안 됩니다.
    pairs = [(x.id, x.text,
              compress(x.text, variant="v2", model_name=model_for("v2"), **params4)[0])
             for x in cs4]
    r = verify.billed(pairs, deployment=DEPLOY, model=DEPLOY, limit=4)
    BILLED = r["totals"]
    t = BILLED

    table(
        ["케이스", "tiktoken 전→후", "API 실측 전→후", "추정 절감", "실측 절감"],
        [[x["id"],
          f'{x["local_before"]:,} → {x["local_after"]:,}',
          f'{x["api_before"]:,} → {x["api_after"]:,}',
          pct(x["local_saved"]), pct(x["api_saved"])]
         for x in r["rows"]],
        foot=["합계",
              f'{t["local_before"]:,} → {t["local_after"]:,}',
              f'{t["api_before"]:,} → {t["api_after"]:,}',
              pct(t["local_saved"]), pct(t["api_saved"])],
        align=["left", "right", "right", "right", "right"],
        title=f'과금 기준으로 다시 재기 ({t["n"]}건)',
        note="'API 실측' 은 응답의 usage.input_tokens 를 그대로 읽은 값입니다.",
    )

    print(verify.verdict(t))
    print(f'텍스트당 오버헤드 {t["overhead_per_text"]:+.1f} 토큰 — '
          f'역할 구분자 같은 프레이밍이라 길이와 무관하게 붙습니다.')
    print(r["counter"].describe())
except Exception as e:
    print(f"과금 검증을 건너뜁니다 — {type(e).__name__}: {str(e)[:160]}")
    print("\\n자격증명이 있으면 아래로 준비하실 수 있습니다.")
    print("  cd labs && cp .env.example .env")
    print("없어도 위까지의 결과는 전부 유효합니다. 다만 추정치입니다.")

## 10. 종합 — 이 셀 하나로 랩 전체 결과 보기

앞의 절들은 축을 하나씩 봤습니다. 여기서는 **모든 축을 한 번에** 돌리고
과금 기준으로 잽니다.

| 축 | 무엇을 |
|---|---|
| 변형 | `v1` · `long` · `v2` |
| 모델 | `small` · `large` |
| 압축률 | `rate` 를 여러 값으로 |
| 언어 | 한국어 · 영어를 따로 집계 |
| 토큰 | **API 응답의 `usage.input_tokens`** — tiktoken 추정이 아닙니다 |

### 비용과 시간

토큰을 실측하므로 텍스트마다 호출이 붙습니다. 조합 수 × 케이스 수만큼
나가지만 **같은 텍스트는 한 번만** 부르고 디스크에 남습니다. 두 번째
실행부터는 압축만 다시 합니다.

처음 돌리시면 몇 분 걸립니다. 줄이시려면 아래 `RATES` 나 `COMBOS` 를
좁히세요.

### 표를 읽는 법

- **절감** — 과금 기준으로 실제로 얼마나 줄었나
- **보존 ko / en** — 답에 필요한 문자열이 남은 비율. 언어별로 따로 봅니다
- **격차** — 영어에서 한국어를 뺀 값. 클수록 언어를 탑니다

**절감이 큰 행이 좋은 게 아닙니다.** 보존율이 함께 높아야 쓸 수 있습니다.

In [ ]:
import itertools

# ── 이 셀의 설정 ──────────────────────────────────────────────
COMBOS = [("v1", "small"), ("v1", "large"),
          ("long", "small"), ("long", "large"),
          ("v2", "small"), ("v2", "large")]
RATES = [0.7, 0.5, 0.3]
RESERVE_DIGIT = True

# 토큰을 API 로 실측합니다. 같은 텍스트는 캐시에서 꺼내 씁니다.
api = T.make_counter({"mode": "api", "cache": True},
                     env.get("AZURE_OPENAI_DEPLOYMENT"))
local = T.make_counter({"mode": "local"}, "gpt-5.4")
print(f"토큰 측정 {api.backend} · 캐시 {getattr(api, 'preloaded', 0)}건 보유")
print(f"조합 {len(COMBOS)} × rate {len(RATES)} × 케이스 {len(cases)} = "
      f"{len(COMBOS) * len(RATES) * len(cases)}회 압축\n")

from kit.provider import ContentFiltered

rows, raw, skipped = [], [], []
for (variant, tier), rate in itertools.product(COMBOS, RATES):
    recs, dropped = [], 0
    for c in cases:
        out, _ = compress(c.text, question=c.question or "", variant=variant,
                          model_name=tier, rate=rate,
                          force_reserve_digit=RESERVE_DIGIT)
        try:
            rec = dict(lang=c.meta["lang"],
                       tb=api(c.text), ta=api(out),
                       lb=local(c.text), la=local(out),
                       s=survival(out, c.must_include))
        except ContentFiltered:
            # 압축 결과가 심하게 깨지면 콘텐츠 필터에 걸립니다. 그 케이스는
            # 빼고 몇 건이 빠졌는지 남깁니다. 로컬 계산으로 슬쩍 바꾸면
            # 한 조합 안에서 측정 방식이 섞여 합계가 뜻을 잃습니다.
            dropped += 1
            continue
        recs.append(rec)
    api.save()

    if dropped:
        skipped.append((variant, tier, rate, dropped))
    if not recs:
        print(f"  {variant:5s}/{tier:5s} rate={rate}  전 케이스 측정 실패 — 건너뜁니다",
              flush=True)
        continue

    tb, ta = sum(r["tb"] for r in recs), sum(r["ta"] for r in recs)
    lb, la = sum(r["lb"] for r in recs), sum(r["la"] for r in recs)
    ko = [r["s"] for r in recs if r["lang"] == "ko"]
    en = [r["s"] for r in recs if r["lang"] == "en"]
    kom, enm = sum(ko) / len(ko), sum(en) / len(en)
    saved = 1 - ta / tb
    raw.append(dict(variant=variant, tier=tier, rate=rate, saved=saved,
                    est=1 - la / lb, ko=kom, en=enm, worst=min(r["s"] for r in recs),
                    tb=tb, ta=ta))
    rows.append([variant, tier, rate, f"{len(recs)}/{len(cases)}",
                 f"{tb:,} → {ta:,}", pct(saved),
                 pct(kom), pct(enm), f"{(enm - kom) * 100:+.0f}%p",
                 pct(min(r["s"] for r in recs))])
    print(f"  {variant:5s}/{tier:5s} rate={rate}  절감 {saved:6.1%} · "
          f"ko {kom:5.0%} · en {enm:5.0%}", flush=True)

table(
    ["변형", "모델", "rate", "측정", "토큰 전→후 (API 실측)", "절감",
     "보존 ko", "보존 en", "격차", "최저"],
    rows,
    align=["left", "left", "right", "right", "right", "right",
           "right", "right", "right", "right"],
    title=f"전체 조합 · 토큰은 API 실측 · 케이스 {len(cases)}건 (ko/en 각 {len(cases)//2})",
    note="'절감' 은 과금 기준입니다. 보존율이 함께 높아야 쓸 수 있습니다. "
         "'격차' 는 영어 − 한국어로, 클수록 언어를 탑니다.",
)
print(api.describe())
if skipped:
    print()
    print("콘텐츠 필터로 측정하지 못한 케이스가 있습니다:")
    for variant, tier, rate, n in skipped:
        print(f"    {variant}/{tier} rate={rate} — {n}건")
    print("    압축 결과가 뜻 없는 글자열이 되면 생깁니다. '측정' 열의 분자가")
    print("    분모보다 작은 행은 그만큼 적은 케이스로 낸 값입니다.")

In [ ]:
# 위 표에서 바로 읽어내기 어려운 것들을 뽑습니다. 값은 전부 방금 잰
# 결과에서 계산합니다 — 문서에 박아 두면 조건이 바뀔 때 틀려집니다.
def show(title, items):
    print(title)
    for x in items:
        print("   ", x)
    print()


ok = [r for r in raw if r["ko"] >= 0.9 and r["en"] >= 0.9]
best_saving = max(raw, key=lambda r: r["saved"])
best_usable = max(ok, key=lambda r: r["saved"]) if ok else None
widest = max(raw, key=lambda r: r["en"] - r["ko"])
gap_est = max(raw, key=lambda r: abs(r["est"] - r["saved"]))


def tag(r):
    return f"{r['variant']}/{r['tier']} rate={r['rate']}"


show("① 가장 많이 줄인 조건", [
    f"{tag(best_saving)} — 절감 {best_saving['saved']:.1%} · "
    f"보존 ko {best_saving['ko']:.0%} / en {best_saving['en']:.0%}",
    "절감만 크고 보존율이 낮으면 쓸 수 없습니다. 아래와 비교해 보세요.",
])

show("② 양쪽 언어 모두 보존 90% 이상을 지킨 조건", (
    [f"{tag(r)} — 절감 {r['saved']:.1%}" for r in
     sorted(ok, key=lambda r: -r["saved"])[:5]] +
    [f"→ 그중 최대 절감은 {tag(best_usable)} 의 {best_usable['saved']:.1%} 입니다"]
) if ok else ["없습니다. 이 코퍼스·설정에서는 90% 를 지키는 조합이 없었습니다."])

show("③ 언어 격차가 가장 큰 조건", [
    f"{tag(widest)} — 영어 {widest['en']:.0%} · 한국어 {widest['ko']:.0%} "
    f"({(widest['en'] - widest['ko']) * 100:+.0f}%p)",
    "전체 평균만 보면 이 격차가 보이지 않습니다.",
])

show("④ tiktoken 추정과 과금 실측의 차이", [
    f"가장 벌어진 조건: {tag(gap_est)} — "
    f"추정 {gap_est['est']:.1%} vs 실측 {gap_est['saved']:.1%} "
    f"({(gap_est['est'] - gap_est['saved']) * 100:+.1f}%p)",
    f"전체 평균 차이 {sum(r['est'] - r['saved'] for r in raw) / len(raw) * 100:+.1f}%p",
    "추정이 절감을 부풀리는 방향이면 실제 청구는 생각보다 덜 줄어듭니다.",
])

print("변형별로 rate 를 올렸을 때 보존율이 어떻게 움직이는지")
for v in ("v1", "long", "v2"):
    for t in ("small", "large"):
        xs = sorted([r for r in raw if r["variant"] == v and r["tier"] == t],
                    key=lambda r: -r["rate"])
        if xs:
            line = " → ".join(f"{r['rate']}:{(r['ko'] + r['en']) / 2:.0%}" for r in xs)
            print(f"    {v:5s}/{t:5s}  {line}")

## 정리 — 이 랩에서 확인하신 것

숫자는 **고른 모델·`rate`·코퍼스에 따라 달라집니다.** 여기서는 무엇을
확인하는 랩이었는지만 정리합니다. 실제 값은 위의 표를 보세요.

- **변형마다 버리는 근거가 다릅니다** — `v1`/`long` 은 토큰별 정보량,
  `v2` 는 학습된 판정입니다. 같은 `rate` 라도 결과가 다른 이유입니다.
- **절감률만으로 판단할 수 없습니다** — 압축을 거의 안 해도 보존율은
  100% 로 나옵니다. 두 값을 항상 같이 보세요.
- **언어가 결과를 바꿉니다** — 같은 내용의 한·영 쌍으로 확인하셨습니다.
  전체 평균은 언어별 격차를 가립니다.
- **`rate=1.0` 이 무손실이 아닐 수 있습니다** — 토큰에서 텍스트를
  재구성하는 방식이면 서식이 바뀌고 토큰이 늘 수도 있습니다.
- **줄인 토큰이 곧 시간 절약은 아닙니다** — 압축 자체가 시간을 씁니다.
- **추정과 과금은 다릅니다** — 마지막 절에서 API 실측으로 확인하셨습니다.

### 다른 조건으로도 돌려보세요

이 랩의 결과는 **모델과 `rate` 에 크게 좌우됩니다.** 위의 드롭다운에서
`large` 를 고르거나 `rate` 를 바꿔 다시 돌리시면, 같은 코드로 전혀 다른
그림이 나옵니다. 논문은 `v1`/`long` 에 7B 를 썼고 이 노트북의 기본값은
0.5B 입니다.